In [41]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Importing CRUD Module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################


username = "aacuser"
password = "password"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
#app = JupyterDash(__name__)

#image loading and encoding
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read()) 

#application start
app = JupyterDash('Adrian Azevedo')


# Because of the nature of this logic I will be citing information from resources such as murach's HTML5 and CSS3 4th 
#edition by Anne Boehm and Zak Ruvalcaba as well as module resources such as Head First Python 2nd edition by Paul Barry

app.layout = html.Div([  #Div in HTML is a page division here we create the inital page using this mechanic.
    
    # The Code below creates a page division which seperates the image from the rest of the code
    #**** href allows you to link the site to the image and html.img creates the size of the image in pixels
    #**** src='data:image/png;base64,{}'.format(encoded_image.decode()), is and the code around it are taken from murach's HTML5 and CSS3 4th
    
    html.Div(    
        
        html.A(
            href="https://www.snhu.edu",
            target="_blank",
            
            children=[
                
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={
                        'height': '200px',
                        'width': 'auto',
                        'display': 'block',
                        'margin': '0 auto'
                        
                    }
                )
            ]
        )
    ),
    
    
    
    #Most of the major parts of this code was prewritten 

    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    html.Hr(),
    # This is part the button implimentation from module 6 resources. it orignally added buttons for cats and dogs 
    # it now makes use of radio items from Head First Python 2nd edition by Paul Barry which allows for a cleaner button implementation.
    
    html.Div([ #divides page for buttons
        dcc.RadioItems(
            id='filter-type',
            # creates button layout
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain Rescue', 'value': 'mountain'},
                {'label': 'Disaster Rescue', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True
        )
    ]),
    
    # the following code is just a straight unmodified rip of module 6

    dash_table.DataTable(
        id='datatable-id',
        # organization of data take also from the module resources.
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        
        # hidden in the code below is the logic for the selectable buttona nd the logic for text searches above each collumn.
        data=df.to_dict('records'),
        
        page_size=10,
        sort_action='native', #this toggles between down up and top down sorting.
        filter_action='native',#search in collumns 
        row_selectable='single',#selectable button
        selected_rows=[0],

        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '180px'
        }
    ),

    html.Br(), # makes a page break
    html.Hr(), # makes a page break
    #map and chart format logic
    html.Div(
    className='row',
    style={'display': 'flex'},
    children=[
        html.Div(
            id='graph-id',
            className='col s12 m6'
        ),
        html.Div(
            id='map-id',
            className='col s12 m6'
            )
        ]
    )


])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'columns')],
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    # due to the nature of this code I will be discussing mongoDB.

    if filter_type == "reset": #this links the button to the logic
        df = pd.DataFrame.from_records(db.read({})) 
        #this makes a general query to MongoDB which reads all queries 
        # this therefore resets the settings back to normal
        df.drop(columns=['_id'], inplace=True)

  
    elif filter_type == "water":
        # this is a dictionary defined in the variable query which is used to retrieve data from the table.
        # it then passes this query through df = pd.DataFrame.from_records(db.read(query)) which links it to the 
        #database
        query = { 
            "breed": 
            {
                "$in": 
                [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": 
            {
                "$gte": 26,
                "$lte": 156
            }
        }
        
        df = pd.DataFrame.from_records(db.read(query))
        df.drop(columns=['_id'], inplace=True)

    # muntain rescue copied logic from above.
    elif filter_type == "mountain":
        query = {
            "breed": 
            {
                "$in": 
                [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": 
            {
                "$gte": 26,
                "$lte": 156
            }
        }
        df = pd.DataFrame.from_records(db.read(query))
        df.drop(columns=['_id'], inplace=True)

    # disater rescue copied logic from above.
    elif filter_type == "disaster":
        query = {
            "breed": 
            {
                "$in": 
                [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": 
            {
                "$gte": 20,
                "$lte": 300
            }
        }
        df = pd.DataFrame.from_records(db.read(query))
        df.drop(columns=['_id'], inplace=True)
    # this code should not apply but what it does is if there is an error in the code above this should make any button a reset.
    else:
        df = pd.DataFrame.from_records(db.read({}))
        df.drop(columns=['_id'], inplace=True)
        
    # this is more formating code.
    columns = [
        {"name": i, "id": i, "deletable": False, "selectable": True}
        for i in df.columns
    ]

    data = df.to_dict("records")
    #returns data
    
    return data, columns

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # allowable breeds logic to prevent complex pie chart
    
    allowed_breeds = [
        "Labrador Retriever Mix",
        "Chesapeake Bay Retriever",
        "Newfoundland",
        "German Shepherd",
        "Alaskan Malamute",
        "Old English Sheepdog",
        "Siberian Husky",
        "Rottweiler",
        "Doberman Pinscher",
        "Golden Retriever",
        "Bloodhound"
    ]
    #good to have chart indicating mix breeds that might be applicable
    mix_allowed_breeds = [
        "Labrador Retriever",
        "Chesapeake Bay Retriever Mix",
        "Newfoundland Mix",
        "German Shepherd Mix",
        "Alaskan Malamute Mix",
        "Old English Sheepdog Mix",
        "Siberian Husky Mix",
        "Rottweiler Mix",
        "Doberman Pinscher Mix",
        "Golden Retriever Mix",
        "Bloodhound Mix"
    ]
    #Non valid Breed elimination logic
    
    valid_breeds = allowed_breeds + mix_allowed_breeds

    dff['breed_group'] = dff['breed'].apply(
        lambda x: x if x in valid_breeds else None
    )
    dff = dff.dropna(subset=['breed_group'])
    px.pie(dff, names='breed_group')

    
    #plotly logic that formats the pie graph.
    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed_group',
                title='Rescue Breeds'
            )
        )
    ]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
#copy pasted logic from module 6
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    row = 0 if not index else index[0]

    # safety check
    if row >= len(dff):
        row = 0

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'}, #Dimensional Data.
            center=[30.75, -97.48], # AUSTIN TEXAS LONGITUDE AND LATIT
            zoom=10, #Inital zoom
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode="inline")